In [1]:
import os
print(os.getcwd())          
print(os.listdir('.'))
import re
import glob
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

c:\Users\LEGION\OneDrive\Documentos\Tec\Plataformas de analítica de negocios\Extracción de características
['10_VGAC.ipynb', '4_Extracción_de_Características.ipynb', 'Analisis GAC Full7agosto 2026(Conversaciones por Asesor).csv', 'Analisis GAC Full7agosto 2026(Principales Canales).csv', 'Analisis GAC Full7agosto 2026(TOPS MKT).csv', 'Base_Datos_GAC', 'charts', 'microretailer_mit_lift_lab.xlsx']


In [2]:
DATA = r'C:\Users\LEGION\OneDrive\Documentos\Tec\Plataformas de analítica de negocios\Extracción de características'
OUT  = './charts/'
os.makedirs(OUT, exist_ok=True)

plt.rcParams['font.size'] = 11
PALETTE = ['#2E5C8A', '#4C8FBF', '#7FB3D5', '#F2A65A', '#E07A5F', '#81B29A', '#3D405B']

In [3]:
def clean_num(x):
    """Convierte celdas tipo '1,234' o '-' a float, o NaN."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace('%', '').replace(',', '')
    if s in ('-', '', 'nan'):
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan

In [4]:
def guardar_bar(df, cat_col, val_col, title, fname, ylabel='', pct=False,
                 color=None, horizontal=False, figsize=(7, 4.2)):
    fig, ax = plt.subplots(figsize=figsize)
    colors = color if color else PALETTE[:len(df)]
    if horizontal:
        ax.barh(df[cat_col], df[val_col], color=colors)
        ax.invert_yaxis()
        ax.set_xlabel(ylabel)
    else:
        ax.bar(df[cat_col], df[val_col], color=colors)
        ax.set_ylabel(ylabel)
        plt.xticks(rotation=30, ha='right')
    ax.set_title(title, fontsize=13, fontweight='bold')
    if pct:
        ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y' if not horizontal else 'x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT + fname, dpi=150)
    plt.close()

In [5]:
def guardar_pie(df, cat_col, val_col, title, fname, figsize=(6, 6)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.pie(df[val_col], labels=df[cat_col], autopct='%1.1f%%',
           colors=PALETTE[:len(df)],
           wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
           textprops={'fontsize': 10})
    ax.set_title(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT + fname, dpi=150)
    plt.close()

In [6]:
asesores = pd.read_csv(os.path.join(DATA, 'Analisis GAC Full7agosto 2026(Conversaciones por Asesor).csv'),
                        encoding='utf-8-sig')
asesores.columns = ['Asesor', 'Conversaciones']
asesores['Asesor'] = asesores['Asesor'].str.strip()
asesores = asesores.dropna()

In [7]:
raw = pd.read_csv('Analisis GAC Full7agosto 2026(Principales Canales).csv',
                   header=None, encoding='utf-8-sig')

def extract_block(month_row_idx, metrics, n_months=30):
    """Extrae un bloque (Piso/Plazas/Digital) a formato largo mes x métrica."""
    months = raw.iloc[month_row_idx, 1:1 + n_months].tolist()
    block = {}
    for label, row_idx in metrics.items():
        vals = raw.iloc[row_idx, 1:1 + n_months].apply(clean_num).tolist()
        block[label] = vals
    df = pd.DataFrame(block)
    df.insert(0, 'Mes', months)
    return df

In [8]:
piso = extract_block(6, {
    'Leads': 7, 'Efectivos': 8, 'SDC': 9, 'PDM': 10,
    'Citas Programadas': 11, 'Citas Efectivas': 12, 'Ventas': 13
})
piso['Canal'] = 'Piso'

plazas = extract_block(19, {
    'Leads': 20, 'Efectivos': 21, 'SDC': 22, 'PDM': 23,
    'Citas Programadas': 24, 'Citas Efectivas': 26, 'Ventas': 28
})
plazas['Canal'] = 'Plazas'

digital = extract_block(34, {
    'Leads': 35, 'Efectivos': 36, 'SDC': 37, 'PDM': 38,
    'Citas Programadas': 39, 'Citas Efectivas': 40, 'Ventas': 41
})
digital['Canal'] = 'Digital'

canales = pd.concat([piso, plazas, digital], ignore_index=True)
canales = canales.dropna(subset=['Mes'])

In [9]:
def get_anio(mes):
    m = re.search(r',\s*(\d{2})', str(mes))
    return 2000 + int(m.group(1)) if m else np.nan

canales['Anio'] = canales['Mes'].apply(get_anio)
canales['MesNombre'] = canales['Mes'].apply(lambda m: str(m).split(',')[0].strip())

trimestre_map = {
    'Enero': 'T1', 'Febrero': 'T1', 'Marzo': 'T1',
    'Abril': 'T2', 'Mayo': 'T2', 'Junio': 'T2',
    'Julio': 'T3', 'Agosto': 'T3', 'Septiembre': 'T3',
    'Octubre': 'T4', 'Noviembre': 'T4', 'Diciembre': 'T4'
}
canales['Trimestre'] = canales['MesNombre'].map(trimestre_map)

semestre_map = {'T1': '1er Semestre', 'T2': '1er Semestre',
                 'T3': '2do Semestre', 'T4': '2do Semestre'}
canales['Semestre'] = canales['Trimestre'].map(semestre_map)

In [10]:
raw2 = pd.read_csv(os.path.join('Analisis GAC Full7agosto 2026(TOPS MKT).csv'),
                    header=None, encoding='utf-8-sig')
meses_cols = raw2.iloc[1, 1:13].tolist()

kpi_blocks = {
    'Afluencia a Piso': 10,
    'Ventas de Plaza': 20,
    'Tendencia Redes Sociales': 31,
    'Utilidad Neta': 40,
    'Rotacion de Personal': 49,
    'Recuperacion de Marca': 59,
}

rows = []
for kpi, ridx in kpi_blocks.items():
    vals = raw2.iloc[ridx, 1:13].apply(clean_num).tolist()
    for mes, v in zip(meses_cols, vals):
        if pd.notna(v):
            rows.append({'Indicador': kpi, 'Mes': mes, 'Cumplimiento_%': v})

kpi_long = pd.DataFrame(rows)
print(kpi_long.head()) 

          Indicador      Mes  Cumplimiento_%
0  Afluencia a Piso    Enero           129.0
1  Afluencia a Piso  Febrero           130.0
2  Afluencia a Piso    Marzo           145.0
3  Afluencia a Piso    Abril           109.0
4  Afluencia a Piso     Mayo           159.0


In [11]:
mask_placeholder = (kpi_long['Indicador'] == 'Utilidad Neta') & (kpi_long['Cumplimiento_%'] == 0)
kpi_long = kpi_long[~mask_placeholder].reset_index(drop=True)

def nivel_cumplimiento(v):
    if v < 100:
        return 'Bajo (<100%)'
    elif v <= 130:
        return 'Medio (100-130%)'
    return 'Alto (>130%)'

kpi_long['Nivel_Cumplimiento'] = kpi_long['Cumplimiento_%'].apply(nivel_cumplimiento)
kpi_long['Objetivo_Alcanzado'] = np.where(kpi_long['Cumplimiento_%'] >= 100, 'Si', 'No')

In [12]:
df = asesores.sort_values('Conversaciones', ascending=False).reset_index(drop=True)
df['Porcentaje'] = (df['Conversaciones'] / df['Conversaciones'].sum() * 100).round(1)
guardar_bar(df, 'Asesor', 'Conversaciones',
            'Conversaciones iniciadas por Asesor (Feb-26)',
            '01_asesor.png', ylabel='N° conversaciones', horizontal=True, figsize=(7, 5))

In [14]:
canal_leads = canales.groupby('Canal')['Leads'].sum().reset_index().sort_values('Leads', ascending=False)
guardar_pie(canal_leads, 'Canal', 'Leads', 'Distribución de Leads por Canal (Feb24-Jul26)', '02_canal_leads.png')

canal_ventas = canales.groupby('Canal')['Ventas'].sum().reset_index().sort_values('Ventas', ascending=False)
guardar_pie(canal_ventas, 'Canal', 'Ventas', 'Distribución de Ventas por Canal (Feb24-Jul26)', '03_canal_ventas.png')

In [15]:
etapas = ['Leads', 'Efectivos', 'SDC', 'PDM', 'Citas Programadas', 'Citas Efectivas', 'Ventas']
etapa_tot = pd.DataFrame({'Etapa': etapas, 'Total': [canales[e].sum() for e in etapas]})
guardar_bar(etapa_tot, 'Etapa', 'Total', 'Volumen total por Etapa del Embudo Comercial',
            '04_embudo.png', ylabel='Total acumulado', color=PALETTE * 2)